# STABLE in a nutshell

A classifier trained under a fairness constraint is usually fair **at the group
proportions of its training data**. Once deployed in a population with a
different composition, its prediction rate can drift away from the level it
was calibrated for.

STABLE views fairness as **distributional stability**: the prediction rate
$\Phi(h;\pi')$ should stay close to a target $t$ for every deployment
proportion $\pi'$ in a range $\Pi = [\bar\pi, 1-\bar\pi]$. It learns a convex
mixture of logistic regressions by solving a second-order cone program.

This notebook trains ERM and STABLE on Adult (demographic parity), compares how
their prediction rates move with $\pi'$, and sweeps the fairness budget.
It takes a few minutes on a laptop.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))   # repository root

import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from stable import ERMBaseline, SOCPFairnessMixture
from stable.data import load_adult
from stable.metrics import evaluate, targets

## Data
Adult Income; the sensitive attribute $G$ is binary. 70/30 train/test split.

In [ ]:
X, G, y = load_adult()
Xtr, Xte, Gtr, Gte, ytr, yte = train_test_split(X, G, y, test_size=0.3, random_state=0)
print(f"n = {len(y)},  P(G=1) = {G.mean():.2f},  P(Y=1) = {y.mean():.2f}")

## ERM and STABLE

The target $t$ is ERM's mean prediction on the training set. STABLE's budget
$\delta$ is expressed through $\delta_0$, the targeted unfairness gap $\mathcal{U}(h)$
(in the paper, $\mathcal{U}(h)\lesssim\delta_0$ up to estimation error): $\delta = \delta_0^2\,\mathrm{Var}_\Pi(\pi)$.

In [ ]:
erm = ERMBaseline().fit(Xtr, Gtr, ytr)
t = targets(erm.predict_proba(Xtr), Gtr, ytr)[0]          # DP target

pi_bar, delta0 = 0.02, 0.02
var_pi = (1 - 2 * pi_bar) ** 2 / 12
stable = SOCPFairnessMixture(pi_bar=pi_bar, delta=delta0**2 * var_pi, t=t,
                             constraint="DP").fit(Xtr, Gtr, ytr, seed=0)
print(f"t = {t:.3f}; STABLE mixes {np.sum(stable.alpha_ > 1e-4)} of "
      f"{len(stable.alpha_)} dictionary components")

## Prediction rate under demographic shift

$\Phi(h;\pi')$ is the mean prediction in a population where a fraction $\pi'$
belongs to group 1. It is linear in $\pi'$: its slope is the unfairness gap,
its distance to $t$ at the extremes $\pi'\in\{0,1\}$ is the deployment gap.

In [ ]:
pis = np.linspace(0, 1, 50)
fig, ax = plt.subplots(figsize=(5, 3.5))
for name, model, color in [("ERM", erm, "crimson"), ("STABLE", stable, "tab:blue")]:
    r = evaluate(model.predict_proba(Xte), Gte, yte, t, "DP")
    ax.plot(pis, (1 - pis) * r["phi0"] + pis * r["phi1"], color=color, lw=2,
            label=f"{name}: acc {r['acc']:.3f}, U {r['u']:.3f}, deploy gap {r['deploy']:.3f}")
ax.axhline(t, color="0.3", ls="--", lw=1, label="target $t$")
ax.set_xlabel(r"deployment proportion $\pi'$")
ax.set_ylabel(r"$\Phi(h;\pi')$")
ax.legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.18))
plt.show()

## Sweeping the fairness budget

A smaller $\delta_0$ flattens the curve and keeps it at $t$, at some cost in
accuracy. The same sweep with every baseline is `experiments/pareto.py`.

In [ ]:
rows = []
for d0 in [0.005, 0.01, 0.02, 0.05, 0.10]:
    m = SOCPFairnessMixture(pi_bar=pi_bar, delta=d0**2 * var_pi, t=t,
                            constraint="DP").fit(Xtr, Gtr, ytr, seed=0)
    rows.append(evaluate(m.predict_proba(Xte), Gte, yte, t, "DP"))
erm_r = evaluate(erm.predict_proba(Xte), Gte, yte, t, "DP")

fig, axes = plt.subplots(1, 2, figsize=(8, 3))
for ax, key, xlabel in [(axes[0], "u", r"$\mathcal{U}(h)$"), (axes[1], "deploy", "deployment gap")]:
    ax.plot([r[key] for r in rows], [r["acc"] for r in rows], "o-", color="tab:blue", label="STABLE")
    ax.scatter(erm_r[key], erm_r["acc"], marker="*", s=150, color="crimson", label="ERM", zorder=3)
    ax.set_xlabel(xlabel + " (lower is better)")
    ax.set_ylabel("accuracy")
axes[0].legend()
plt.tight_layout()
plt.show()